# T8 It4: the pre-registered finalists, scored once on official TEST and the blind NUDT 707 draw

Everything scored here was written to `results/preregistered_t8.json` (It3) before this notebook was run: the rules, their numbers, the
attitude model and the success criterion. Run once; results are saved, not tuned. The judge (T7 architecture) is the NUDT-only one, as the
team's review asks: trained on the five NUDT design sets, cut chosen on them.

In [ ]:
import os, sys, time, json, hashlib
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
from train.checks.candidate_judge import FEATURES_CTX, baseline, decide, judge_probability, paired_gain, summary
from train.checks.holdout import NUDT_DESIGN, answer_change, attitude_frame, bar_frontier, load_t8_table, versus_frontier, with_attitude
from train.checks.two_tier import decide_two_tier
RES = ROOT / "tests/Test_Iteration_8/results"
PRE_PATH = RES / "preregistered_t8.json"
print("pre-registration", PRE_PATH.stat().st_mtime, "sha256", hashlib.sha256(PRE_PATH.read_bytes()).hexdigest())
PRE = json.loads(PRE_PATH.read_text())
FINAL = PRE["final_sets"]
table = load_t8_table(ROOT, locked=False, blind=True)      # the only notebook that loads official_test and nudt707
table.groupby("data").photo.nunique()

In [ ]:
t0 = time.time()
pj = judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm")[0]          # trained on NUDT design sets; plain predictions on the final sets
J = lambda d: dict(override=d["override"], min_choice=d["min_choice"], wide_tau=d["wide_tau"])
pc, jet = PRE["pc"], PRE["jetson"]
R = {("pc", "base: bar 8 / 11"): baseline(table, 8, 11, min_choice=3),
     ("pc", "T7 judge, NUDT only"): decide(table, pj, pc["judge_nudt_only"]["tau"], **J(pc["judge_nudt_only"])),
     ("pc", "two-tier"): decide_two_tier(table, pc["two_tier"], "pc"),
     ("pc", "two-tier, a_low 4"): decide_two_tier(table, pc["two_tier_a4"], "pc"),
     ("jetson", "base: Jetson today"): baseline(table, 8, 8, jetson=True, m=5),
     ("jetson", "T7 judge, NUDT only"): decide(table, pj, jet["judge_nudt_only"]["tau"], jetson=True, m=5, **J(jet["judge_nudt_only"])),
     ("jetson", "T7 judge, NUDT only + early exit"): decide(table, pj, jet["judge_nudt_only"]["tau"], jetson=True, m=5, early_tau=0.98, **J(jet["judge_nudt_only"])),
     ("jetson", "two-tier"): decide_two_tier(table, jet["two_tier"], "jetson"),
     ("jetson", "two-tier + early exit"): decide_two_tier(table, jet["two_tier_early"], "jetson")}
R = {k: v[v.data.isin(FINAL)].reset_index(drop=True) for k, v in R.items()}
print("%.0f s" % (time.time() - t0))

## 1. Decision rules

In [ ]:
final_tab = table[table.data.isin(FINAL)].reset_index(drop=True)
FRONT = bar_frontier(final_tab); FRONT.to_csv(RES / "it4_bar_frontier_final.csv", index=False)
rows = []
for (var, name), r in R.items():
    base = R[(var, "base: bar 8 / 11" if var == "pc" else "base: Jetson today")]
    g = paired_gain(base, r, FINAL)
    v = versus_frontier(r, FRONT, var).set_index("data"); a = answer_change(base, r).set_index("data")
    for d in FINAL:
        rows.append(dict(var=var, method=name, data=d, score=v.loc[d, "score"], wrong=v.loc[d, "wrong"], answered=v.loc[d, "n_answered"], queries=v.loc[d, "queries"],
                         gain=g[d], above_bar=v.loc[d, "above_bar"], same_n_bar=v.loc[d, "same_n_bar"], bar_wrong=v.loc[d, "bar_wrong"],
                         added_right=a.loc[d, "added_right"], added_wrong=a.loc[d, "added_wrong"], removed=a.loc[d, "removed"], removed_wrong=a.loc[d, "removed_wrong"]))
OUT = pd.DataFrame(rows); OUT.to_csv(RES / "it4_final.csv", index=False)
pd.concat({k: v for k, v in R.items()}, names=["var", "method"]).reset_index(level=[0, 1]).to_csv(RES / "it4_final_photos.csv", index=False)
display(OUT.set_index(["data", "var", "method"]).sort_index(level=0, sort_remaining=False).round(2))

## 2. Attitude correction on official TEST (model fitted on TRY in It3, applied unchanged)

In [ ]:
ATT = dict(terms=tuple(PRE["attitude"]["terms"]), coef=np.array(PRE["attitude"]["coef"]), mirror=PRE["attitude"]["mirror"])
DUMP = str(ROOT / "tests/Test_Iteration_6/results/rich_dump/official")
rows = []
for (var, name), r in R.items():
    off = r[r.data == "official_test"]
    fixed = with_attitude(off, attitude_frame(off, DUMP), ATT)
    g = paired_gain(off, fixed, ["official_test"])["official_test"]
    ans = off.stage != "skip"
    rows.append(dict(var=var, method=name, score=6 * off.pts.mean(), with_attitude=6 * fixed.pts.mean(), gain=g,
                     within1_before=(off.err_sigma[ans] <= 1).mean(), within1_after=(fixed.err_sigma[ans] <= 1).mean(), wrong_after=int((fixed.pts < 0).sum())))
ATT_OUT = pd.DataFrame(rows); ATT_OUT.to_csv(RES / "it4_attitude_test.csv", index=False); display(ATT_OUT.round(3))

## 3. Against the pre-registered success criterion

In [ ]:
def verdict(var, name):
    base = OUT[(OUT["var"] == var) & OUT.method.str.startswith("base")].set_index("data")
    m = OUT[(OUT["var"] == var) & (OUT.method == name)].set_index("data")
    lo = {d: float(m.loc[d, "gain"].split("[")[1].split(",")[0]) for d in FINAL}
    mean = {d: float(m.loc[d, "gain"].split(" ")[0]) for d in FINAL}
    safe = all(m.loc[d, "wrong"] <= base.loc[d, "wrong"] for d in FINAL)
    return dict(var=var, method=name, no_more_wrong=safe, gain_nonneg=all(mean[d] >= 0 for d in FINAL), interval_above_0=any(lo[d] > 0 for d in FINAL),
                counts=safe and all(mean[d] >= 0 for d in FINAL) and any(lo[d] > 0 for d in FINAL))
V = pd.DataFrame([verdict(v, n) for (v, n) in R if not n.startswith("base")]); V.to_csv(RES / "it4_verdict.csv", index=False); V